## 4. Conversation History with a File Provider

**Goal**: Use `FileHistoryProvider` to save a simple conversation to a local JSON Lines (`.jsonl`) file, then read and display its contents. No function tools are needed.

Run the cells in order with the project's `.venv` kernel. Configure `AZURE_AI_PROJECT_ENDPOINT_FULL` and `AZURE_AI_MODEL_DEPLOYMENT_NAME` in `.env`, and authenticate with `az login`.

The provider is experimental. History is stored as **plaintext**, not encrypted or redacted. Use only non-sensitive demo messages. Files remain in the local `sessions` folder after the notebook finishes; that folder is excluded from Git.

In [1]:
# Copyright (c) Microsoft. All rights reserved.

import json
import os
import sys
from pathlib import Path

sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from dotenv import load_dotenv
from agent_framework import Agent, FileHistoryProvider
from agent_framework.foundry import FoundryChatClient

load_dotenv()

from utils.config import get_env, get_credential

client = FoundryChatClient(
    project_endpoint=get_env("AZURE_AI_PROJECT_ENDPOINT_FULL"),
    model=get_env("AZURE_AI_MODEL_DEPLOYMENT_NAME"),
    credential=get_credential(),
)

project_directory = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
storage_directory = project_directory / "sessions"
storage_directory.mkdir(parents=True, exist_ok=True)
print(f"Storage directory: {storage_directory.resolve()}")

c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


2026-10-26 11:29:08,551 - azure.ai.agentserver - INFO - Application Insights not configured, telemetry export disabled.
Storage directory: C:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\sessions


### Part A - Create an agent with file-backed history

The provider loads earlier messages before each run and appends the new user and assistant messages afterward. `store=False` disables server-side response storage so this demo uses local history.

The session has an automatically generated UUID, used as the filename. Rerunning this cell starts a new session and creates a new file on the next successful turn; it does not overwrite previous conversations.

In [2]:
agent = Agent(
    client=client,
    name="FileHistoryAgent",
    instructions="You are a friendly assistant. Keep your answers brief.",
    context_providers=[FileHistoryProvider(storage_directory)],
    default_options={"store": False},
)
session = agent.create_session()
history_file = storage_directory / f"{session.session_id}.jsonl"

print(f"Session ID: {session.session_id}")
print(f"History file: {history_file.resolve()}")

Session ID: f3c15b04-d016-4325-983f-c11e84b0130f
History file: C:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\sessions\f3c15b04-d016-4325-983f-c11e84b0130f.jsonl


C:\Users\dkankalia\AppData\Local\Temp\ipykernel_48312\1236109461.py:5: ExperimentalWarning: [FILE_HISTORY] FileHistoryProvider is experimental and may change or be removed in future versions without notice.


### Part B - Start a conversation and inspect the file

After the first response, the file should contain the user message and the assistant response, one JSON object per line.

In [3]:
query = "My name is Alice and my favorite color is blue. Please remember these details."
response = await agent.run(query, session=session)
print(f"User: {query}")
print(f"Assistant: {response.text}")

print("\nRaw JSONL after the first turn:")
print(history_file.read_text(encoding="utf-8"))

User: My name is Alice and my favorite color is blue. Please remember these details.
Assistant: I'm sorry, but I can't store personal information. However, I can remember it while we're chatting. How can I assist you today, Alice?

Raw JSONL after the first turn:
{"type": "message", "role": "user", "contents": [{"type": "text", "text": "My name is Alice and my favorite color is blue. Please remember these details.", "additional_properties": {}}], "additional_properties": {}}
{"type": "message", "role": "assistant", "contents": [{"type": "text", "text": "I'm sorry, but I can't store personal information. However, I can remember it while we're chatting. How can I assist you today, Alice?", "additional_properties": {}}], "author_name": "FileHistoryAgent", "additional_properties": {}}



### Part C - Ask a follow-up using the same session

The provider supplies the saved conversation to this call. The assistant should answer **Alice** and **blue** without needing those details repeated in the new question. Exact responses may vary.

In [4]:
query = "What is my name and my favorite color?"
response = await agent.run(query, session=session)
print(f"User: {query}")
print(f"Assistant: {response.text}")

User: What is my name and my favorite color?
Assistant: Your name is Alice, and your favorite color is blue.


### Part D - Display the persisted conversation

The raw text shows the exact file contents. The formatted records show the same data with indentation for readability; the file itself stays compact JSONL. With these two plain-text turns, expect four message records: user, assistant, user, assistant.

You can also open the printed file path in VS Code. Rerun this display cell at any time after a completed turn to inspect the current history.

In [5]:
raw_history = history_file.read_text(encoding="utf-8")
records = [json.loads(line) for line in raw_history.splitlines() if line.strip()]

print(f"History file: {history_file.resolve()}")
print(f"Stored messages: {len(records)}")
print("\n=== Raw JSONL ===")
print(raw_history)

print("=== Formatted message records ===")
for index, record in enumerate(records, start=1):
    print(f"\nMessage {index}:")
    print(json.dumps(record, indent=2, ensure_ascii=False))

History file: C:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\sessions\f3c15b04-d016-4325-983f-c11e84b0130f.jsonl
Stored messages: 4

=== Raw JSONL ===
{"type": "message", "role": "user", "contents": [{"type": "text", "text": "My name is Alice and my favorite color is blue. Please remember these details.", "additional_properties": {}}], "additional_properties": {}}
{"type": "message", "role": "assistant", "contents": [{"type": "text", "text": "I'm sorry, but I can't store personal information. However, I can remember it while we're chatting. How can I assist you today, Alice?", "additional_properties": {}}], "author_name": "FileHistoryAgent", "additional_properties": {}}
{"type": "message", "role": "user", "contents": [{"type": "text", "text": "What is my name and my favorite color?", "additional_properties": {}}], "additional_properties": {}}
{"type": "message", "role": "assistant", "contents": [{"type": "text", "text": "Your name is Alice, and your favorite color is blue.", "a

In [ ]:
## Cosmos History Provider (Not working. How to use it)

# 1. Create a Cosmos history provider instance.
import asyncio
import os
from agent_framework import Agent
from agent_framework.azure import CosmosHistoryProvider
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("FOUNDRY_MODEL")
cosmos_endpoint = os.getenv("AZURE_COSMOS_ENDPOINT")
cosmos_database_name = os.getenv("AZURE_COSMOS_DATABASE_NAME")
cosmos_container_name = os.getenv("AZURE_COSMOS_CONTAINER_NAME")
cosmos_key = os.getenv("AZURE_COSMOS_KEY")

CosmosHistoryProvider(
    endpoint=cosmos_endpoint,
    database_name=cosmos_database_name,
    container_name=cosmos_container_name,
    credential=cosmos_key or credential,
) as history_provider

# 2. Create an agent that uses Cosmos for persisted conversation history.
Agent(
    client=FoundryChatClient(
        project_endpoint=project_endpoint,
        model=model,
        credential=credential,
    ),
    name="CosmosHistoryAgent",
    instructions="You are a helpful assistant that remembers prior turns.",
    context_providers=[history_provider],
    default_options={"store": False},
) as agent

In [ ]:
### Custom History Provider

class CustomHistoryProvider(HistoryProvider):
    """Implementation of custom history provider.
    In real applications, this can be an implementation of relational database or vector store."""

    def __init__(self) -> None:
        super().__init__("custom-history")
        self._storage: dict[str, list[Message]] = {}

    async def get_messages(
        self, session_id: str | None, *, state: dict[str, Any] | None = None, **kwargs: Any
    ) -> list[Message]:
        key = session_id or "default"
        return list(self._storage.get(key, []))

    async def save_messages(
        self,
        session_id: str | None,
        messages: Sequence[Message],
        *,
        state: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> None:
        key = session_id or "default"
        if key not in self._storage:
            self._storage[key] = []
        self._storage[key].extend(messages)

### Associate the custom history provider with the agent.

    agent = Agent(
        client=OpenAIChatClient(),
        name="CustomBot",
        instructions="You are a helpful assistant that remembers our conversation.",
        # Use custom history provider.
        # If not provided, the default in-memory provider will be used.
        context_providers=[CustomHistoryProvider()],
    )